In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm
import random
import json
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from pathlib import Path

load_dotenv()
client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"
)

persist_directory = "/workspaces/Arch_PC_Assistent/embed_model"
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vectorstore = Chroma(embedding_function=embeddings,
                     persist_directory=persist_directory)

In [ ]:
db_data = vectorstore.get(include=["documents","metadatas"])
all_texts = db_data['documents']
all_metadatas = db_data['metadatas']
prompt_count = 498
zufalls_indizes = random.sample(range(len(all_texts)), prompt_count)

neue_fragen_liste = []

print(f"\nGeneriere {prompt_count} synthetische Fragen über DeepSeek...")

# 4. Die Generierungs-Schleife
for idx in tqdm(zufalls_indizes, desc="Prompt generation"):
    chunk_text = all_texts[idx]
    
    prompt = f"""You are a user asking for technical help on a forum. 
    Read the following reference text.
    
    REFERENCE TEXT:
    {chunk_text}
    
    Task: Write a SINGLE, realistic troubleshooting question or "how-to" question that can be answered EXACTLY by using the information in this reference text. 
    Do not mention "the text", "the context", or "the reference" in your question. Ask it naturally, as if you are facing a problem or trying to achieve a specific goal.
    
    Return ONLY the question string, nothing else."""
    
    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.8
        )
        generierte_frage = response.choices[0].message.content.strip()
        generierte_frage = generierte_frage.strip('"').strip("'") 
        
        neue_fragen_liste.append(generierte_frage)
        
    except Exception as e:
        print(f"API Fehler: {e}")

datei_pfad = "/workspaces/Arch_PC_Assistent/dataset/synthetische_rag_fragen.json"
gesammelte_fragen = []


if os.path.exists(datei_pfad):
    try:
        with open(datei_pfad, "r", encoding="utf-8") as f:
            gesammelte_fragen = json.load(f)
    except json.JSONDecodeError:
        print("Datei war leer oder fehlerhaft. Beginne mit einer neuen Liste.")

gesammelte_fragen.extend(neue_fragen_liste)

with open(datei_pfad, "w", encoding="utf-8") as f:
    json.dump(gesammelte_fragen, f, ensure_ascii=False, indent=4)

print(f"\n succesful saved! The file contains {len(gesammelte_fragen)} prompts.")

In [ ]:
alte_datei_pfad = "/workspaces/Arch_PC_Assistent/dataset/arch_v1_dataset.json"
neue_datei_pfad = "/workspaces/Arch_PC_Assistent/dataset/synthetische_rag_fragen.json"
master_datei_pfad = "/workspaces/Arch_PC_Assistent/dataset/master_fragen_liste.json"

master_fragen = []


try:
    with open(alte_datei_pfad, "r", encoding="utf-8") as f:
        alte_daten = json.load(f)
        

    alte_fragen = [item["instruction"] for item in alte_daten if "instruction" in item][:500]    
    master_fragen.extend(alte_fragen)
    print(f"{len(alte_fragen)} old prompt succesfuly extracted.")
except FileNotFoundError:
    print(f"Data '{alte_datei_pfad}' not found.")


try:
    with open(neue_datei_pfad, "r", encoding="utf-8") as f:
        neue_fragen = json.load(f) 
        
    master_fragen.extend(neue_fragen)
    print(f"{len(neue_fragen)} new prompts succesfuly added.")
except FileNotFoundError:
    print(f"Data '{neue_datei_pfad}' not found")


print(f"\nSave {len(master_fragen)} Prompts into gold_dataset...")
with open(master_datei_pfad, "w", encoding="utf-8") as f:
    json.dump(master_fragen, f, ensure_ascii=False, indent=4)

print(f"🎉 FERTIG! Deine '{master_datei_pfad}' ist bereit für den großen Generierungs-Lauf!")